# CSU steering — Colab runner
Thin runner: all code lives in `src/`. Loop: edit locally → commit → push → run cell 1 to pull → run a milestone cell.
Runtime → Change runtime type → GPU (L4 / A100 / T4).

In [ ]:
# 1. Get the latest code
import os
REPO = 'https://github.com/ntmphu/Predicting-Activation-Steering-Failure-from-CSU.git'
DIR = '/content/csu'
if not os.path.exists(DIR):
    !git clone -q {REPO} {DIR}
%cd {DIR}
!git pull -q && git log --oneline -1

In [ ]:
# 2. Install deps (Colab already has a CUDA torch; don't replace it)
!grep -v '^torch' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 3. Google Drive for backups (Colab disconnects lose /content)
from google.colab import drive
drive.mount('/content/drive')
BACKUP = '/content/drive/MyDrive/csu-results'
os.makedirs(BACKUP, exist_ok=True)
# restore earlier results, if any
!cp -rn {BACKUP}/. results/ 2>/dev/null; ls results

In [ ]:
def backup():
    """Copy results/ to Drive. Run after every milestone."""
    !cp -r results/. {BACKUP}/
    print('backed up to', BACKUP)

## Milestones

In [ ]:
# Milestone 1: data split + prompt check
!python -m src.data --model Qwen/Qwen2.5-1.5B-Instruct